# Foundations II: Evaluation and statistical validation

Companion to `slides/slides_01.md`.

The models used here (a decision tree and logistic regression) are treated as **black boxes**; we study them later in the course.
The question today is: **how do we know whether a model is good?**

* **D4a** Why we hold out data.
* **D4b** A single test score is a random variable.
* **D4c** Cross-validation and confidence intervals.
* **D4d** Always compare against a baseline, and choose the right metric.
* **D4e** Comparing two models properly with cross-validation.
* **D4f** Comparing two models on a single test set: McNemar's test.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
plt.rcParams["figure.figsize"] = (9, 4)

X, y = load_breast_cancer(return_X_y=True)
print(f"Breast cancer dataset: {X.shape[0]} samples, {X.shape[1]} features, classes = {np.bincount(y)}")

## D4a — Why we hold out data

Score a model on the data it was trained on, then on data it has never seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
tree = DecisionTreeClassifier(random_state=SEED).fit(X_train, y_train)
print(f"Decision tree — training accuracy: {tree.score(X_train, y_train):.3f}")
print(f"Decision tree — test accuracy    : {tree.score(X_test, y_test):.3f}")

Training accuracy measures **memorization**. Only unseen data measures **generalization**.

## D4b — A single test score is a random variable

Repeat the same experiment with 50 different random train/test splits.

In [ ]:
scores = []
for seed in range(50):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=seed)
    scores.append(DecisionTreeClassifier(random_state=SEED).fit(Xtr, ytr).score(Xte, yte))
scores = np.array(scores)

print(
    f"Test accuracy over 50 splits: min={scores.min():.3f}  max={scores.max():.3f}  mean={scores.mean():.3f}  std={scores.std():.3f}"
)
plt.hist(scores, bins=12, edgecolor="k")
plt.axvline(scores.mean(), color="r", ls="--", label="mean")
plt.xlabel("test accuracy")
plt.ylabel("count")
plt.title("Same model, same data, different random split")
plt.legend()
plt.show()

Reporting "my model has 95% accuracy" from **one** split could be off by several points, just from which samples landed in the test set.

## D4c — k-fold cross-validation and confidence intervals

Each sample is used for testing exactly once. We report **mean ± uncertainty**, not a single number.

In [ ]:
def summarize(name, s, confidence=0.95):
    """Mean with a t-based confidence interval (an approximation: CV folds are not fully independent)."""
    half = stats.t.ppf((1 + confidence) / 2, len(s) - 1) * s.std(ddof=1) / np.sqrt(len(s))
    print(f"{name:28s} {s.mean():.3f} ± {half:.3f}  (95% CI, {len(s)} folds)")


cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
tree_cv = cross_val_score(DecisionTreeClassifier(random_state=SEED), X, y, cv=cv10)
summarize("Decision tree, 10-fold CV", tree_cv)

## D4d — Baselines and the choice of metric

A "dumb" model that always predicts the majority class is the **minimum bar** any model must beat.

In [ ]:
dummy_cv = cross_val_score(DummyClassifier(strategy="most_frequent"), X, y, cv=cv10)
summarize("Majority-class baseline", dummy_cv)

On imbalanced problems, accuracy can be badly misleading. Consider a disease that affects 2% of patients:

In [ ]:
X_imb, y_imb = make_classification(n_samples=5000, weights=[0.98], flip_y=0, random_state=SEED)
Xtr, Xte, ytr, yte = train_test_split(X_imb, y_imb, test_size=0.3, stratify=y_imb, random_state=SEED)
y_pred = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr).predict(Xte)

print("Baseline that says 'healthy' to everyone:")
print(f"  accuracy          = {accuracy_score(yte, y_pred):.3f}")
print(f"  balanced accuracy = {balanced_accuracy_score(yte, y_pred):.3f}")
print(f"  F1 (sick class)   = {f1_score(yte, y_pred, zero_division=0):.3f}")

98% accuracy, and not a single sick patient detected. **The metric must match the goal.**

## D4e — Comparing two models properly

Is logistic regression better than the decision tree, or is the difference just noise?

* Use **the same folds** for both models, then compare the **paired** differences.
* Repeat CV several times (here 5 × 10 folds) to stabilize the estimate.
* Folds share training data, so a naive t-test is over-confident. The **corrected resampled t-test** (Nadeau & Bengio, 2003) inflates the variance to compensate.

In [ ]:
rcv = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=SEED)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
s_tree = cross_val_score(DecisionTreeClassifier(random_state=SEED), X, y, cv=rcv)
s_logreg = cross_val_score(logreg, X, y, cv=rcv)
summarize("Decision tree (5x10 CV)", s_tree)
summarize("Logistic regression (5x10 CV)", s_logreg)


def corrected_resampled_ttest(a, b, test_fraction):
    d = a - b
    k = len(d)
    var = d.var(ddof=1) * (1 / k + test_fraction / (1 - test_fraction))
    t = d.mean() / np.sqrt(var)
    return t, 2 * stats.t.sf(abs(t), k - 1)


t_naive, p_naive = stats.ttest_rel(s_logreg, s_tree)
t_corr, p_corr = corrected_resampled_ttest(s_logreg, s_tree, test_fraction=1 / 10)
print(f"\nMean difference (logreg - tree): {np.mean(s_logreg - s_tree):+.3f}")
print(f"Naive paired t-test     : p = {p_naive:.2e}")
print(f"Corrected resampled test: p = {p_corr:.2e}")

In [ ]:
plt.boxplot([dummy_cv, s_tree, s_logreg], tick_labels=["baseline", "decision tree", "logistic regression"])
plt.ylabel("accuracy")
plt.title("Always report distributions, not single numbers")
plt.show()

## D4f — McNemar's test: two models, one test set

Sometimes cross-validation is too expensive and we only have **one** test set. McNemar's test looks only at the samples where the two models **disagree**:

| | B correct | B wrong |
|---|---|---|
| **A correct** | $a$ | $b$ |
| **A wrong** | $c$ | $d$ |

* $a$ and $d$ (both right, both wrong) say nothing about which model is better.
* If the models were equally good, the disagreements would split evenly: $b \approx c$.

$$\chi^2 = \frac{(|b - c| - 1)^2}{b + c}$$

Under "equally good", $\chi^2$ follows a chi-squared distribution with 1 degree of freedom. For small $b + c$ we use the exact binomial test instead.

In [ ]:
tree_pred = DecisionTreeClassifier(random_state=SEED).fit(X_train, y_train).predict(X_test)
logreg_pred = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train).predict(X_test)
tree_ok, logreg_ok = tree_pred == y_test, logreg_pred == y_test

a = int(np.sum(logreg_ok & tree_ok))
b = int(np.sum(logreg_ok & ~tree_ok))
c = int(np.sum(~logreg_ok & tree_ok))
d = int(np.sum(~logreg_ok & ~tree_ok))
print(f"{'':24s}{'tree correct':>14s}{'tree wrong':>12s}")
print(f"{'logistic reg. correct':24s}{a:14d}{b:12d}")
print(f"{'logistic reg. wrong':24s}{c:14d}{d:12d}")

chi2 = (abs(b - c) - 1) ** 2 / (b + c)
p_chi2 = stats.chi2.sf(chi2, df=1)
p_exact = stats.binomtest(b, b + c, 0.5).pvalue
print(f"\nTest accuracy: logistic regression = {logreg_ok.mean():.3f}, tree = {tree_ok.mean():.3f}")
print(f"McNemar chi^2 = {chi2:.2f}  ->  p = {p_chi2:.4f}")
print(f"Exact binomial version       ->  p = {p_exact:.4f}")

**Takeaways**

1. Never evaluate on training data.
2. A score is an **estimate** with uncertainty: report mean ± CI.
3. Beat a **baseline** with a **metric that matches the goal**.
4. Compare models on the **same folds** with a test that accounts for overlap between folds (corrected t-test), or on the **same test set** with McNemar's test.

Each later topic adapts this protocol: regression metrics, clustering without labels, time series, and so on.